# S5.M12-1 列采样配置的 SGB 训练随机性

**目的**：方法筛选第六阶段第 2 步（[DR-M5-017](../../../registry/decision_records/DR-M5-017.yaml) · D-4 ~ D-5、D-8）。

S5.M2 / S5.M9 的 SGB 上 I2 + I3（优化版联邦 − 原版联邦）都是 seed 11 单次拟合，区间只含测试集抽样，不含训练随机性。
本步在优化版联邦配置列采样 < 1 的全部 9 个任务（按配置判定，不看结果）上补齐多 seed：
UCI、mock 重合 20 / 50、FedAds、Santander G3、Orange 升级、IEEE 第 1 块、Orange 大版偏好 / 追加销售。

**做法**：
- 先做原版联邦的确定性核对：UCI 原版联邦（列、行采样都为 1）换 seed 22 再拟合，与存档的 seed 11 逐位比较；
  - 核对未通过（见第 1 节），按 D-4 预写分支与 D-8，两个方法都在 seed 11 之外再拟合 seed 22 / 33 / 44 / 55 / 66，配置不变；
- 用主动方明文直方图版本（S5.M7）。每个（任务，方法）第一个新 seed 前抽查两项：
  - 前 2 棵树原版与明文逐位相同；
  - 按明文前 2 / 4 棵外推单个 seed 的秒数，超过 7,000 秒的不做；
- 每个 seed 分别存档、分段运行，可续跑；先做全部任务的优化版联邦，再做原版联邦；
- 对比：两层 bootstrap（测试客户簇 × 两个方法各自的 6 个 seed，1,000 次），点估计为两边 6 个 seed 平均测试 AUROC 之差；同口径另报 seed 11 单次。

**数据**：公开数据，属开发证据。

In [1]:
import hashlib
import json
import logging
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
SECONDS_PER_HOUR = 3600
logging.getLogger().setLevel(logging.ERROR)        # 隐语运行日志不进 notebook
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m12_sgb_seeds.yaml"
m12 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
cfg = yaml.safe_load((REPO_ROOT / m12["base_config"]).read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("sgb_seeds.py", "sgb_i2i3_confirm.py", "sgb_cost_screen.py", "method_screen_sgb.py", "method_screen.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
for p in [CONFIG_PATH, REPO_ROOT / m12["decision_record"]] + COMPONENTS:
    print(p.relative_to(REPO_ROOT), "sha256:" + sha(p))
print("决策记录:", m12["decision_record"], "| step:", m12["step_id"], "|", m12["purpose"])
print("seeds:", m12["seeds"], "|", m12["seeds_note"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
M9, SGB, LARGE, M12 = (pathlib.Path(os.environ[m12[k]]) for k in ("m9_dir_env", "sgb_dir_env", "large_dir_env", "out_dir_env"))
DIRS = [M9, SGB]
TASKS = [t["name"] for t in m12["tasks"]]
launch, n_seg = {}, 0
for line in (M12 / "fingerprint_launch.txt").read_text(encoding="utf-8").splitlines():
    if line.startswith("=="):
        n_seg += 1
    elif "sha256:" in line:
        path, digest = line.split(" sha256:")
        launch.setdefault(pathlib.Path(path).name, set()).add(digest)
print("启动指纹：%d 段运行，每段记录的文件与 sha256（路径只留文件名）：" % n_seg)
for name, digests in sorted(launch.items()):
    print("  %s %s" % (name, ", ".join("sha256:" + d[:HASH_PREFIX_LEN] for d in sorted(digests))))
for p in COMPONENTS + [CONFIG_PATH]:
    assert launch[p.name] == {hashlib.sha256(p.read_bytes()).hexdigest()}, p.name
print("各段运行时的组件与配置与当前仓库逐字节相同（决策记录在运行后追加了复核，sha256 以上一行为准）。")

modules/m5_modeling/configs/s5_m12_sgb_seeds.yaml sha256:ed4e94f6c875
registry/decision_records/DR-M5-017.yaml sha256:ff0566dda2d3
modules/m5_modeling/components/sgb_seeds.py sha256:969f168fcd55
modules/m5_modeling/components/sgb_i2i3_confirm.py sha256:7b2d6ed9a322
modules/m5_modeling/components/sgb_cost_screen.py sha256:cf6088aa95be
modules/m5_modeling/components/method_screen_sgb.py sha256:2e54c618305f
modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
决策记录: registry/decision_records/DR-M5-017.yaml | step: S5.M12 | 优化版联邦列采样 < 1 的 9 个任务上，SGB 的 I2 + I3 改用 6 个 seed（两层 bootstrap），与 seed 11 单次结论对照；原版联邦先做确定性核对，未通过则也用 6 个 seed（D-8）
seeds: [11, 22, 33, 44, 55, 66] | 优化版联邦用全部 6 个 seed（seed 11 为已存档的单次拟合，本步新拟合 22 / 33 / 44 / 55 / 66）；原版联邦列、行采样都为 1，但 seed 还经被动方分桶洗牌影响增益并列时的切点选择，确定性核对未通过，按 D-4 / D-8 也用同样 6 个 seed
git: 2ed1ffc（工作区含本步未提交改动）
启动指纹：22 段运行，每段记录的文件与 sha256（路径只留文件名）：
  DR-M5-017.yaml sha256:f8e59289020e
  method_screen.py sha256:05e19e025d0b
  method_screen_sgb.py sha256:

## 1. 运行概况

下一格先列出确定性核对的结果，再列出每个（任务，方法）的抽查与拟合信息：
- 前 2 棵逐位抽查的结果；
- 单个 seed 的外推秒数；
- 5 个新 seed 的实际拟合秒数合计，以及实际与外推之比。

In [2]:
from modules.m5_modeling.components import sgb_seeds as SS

P = SS.MS.Params(cfg)
SEEDS = [int(s) for s in P.seeds[1:]]
det = json.loads((M12 / "determinism.json").read_text(encoding="utf-8"))
methods = SS.methods_for(det)
summary = json.loads((M12 / "summary.json").read_text(encoding="utf-8"))
EVAL = {n: json.loads((M12 / ("%s__seeds_eval.json" % n)).read_text(encoding="utf-8")) for n in TASKS
        if (M12 / ("%s__seeds_eval.json" % n)).exists()}
print("确定性核对（%s · 原版联邦 seed %d 对存档的 seed 11）：逐位相同 %s；%d 个测试样本打分不同，最大差 %.4f；拟合 %.0f 秒" % (
    det["task"], det["seed"], det["identical"], det["n_rows_differ"], det["max_abs_diff"], det["fit_seconds"]))
print("拟合新 seed %s 的方法：%s" % (SEEDS, " / ".join(methods)))
rows = []
for n in TASKS:
    for m in methods:
        pr = json.loads(SS.probe_path(M12, n, m).read_text(encoding="utf-8"))
        fits = [json.loads(SS.seed_path(M12, n, m, sd, "json").read_text(encoding="utf-8"))["fit_seconds"] for sd in SEEDS
                if SS.seed_path(M12, n, m, sd, "json").exists()]
        rows.append({"任务": n, "方法": m, "前 2 棵逐位一致": pr["identical_first_trees"],
                     "单次外推秒数": round(pr["projected_fit_seconds"], 1), "新 seed 数": len(fits),
                     "新 seed 合计秒数": round(sum(fits), 1),
                     "实际 / 外推": round(float(np.mean(fits)) / pr["projected_fit_seconds"], 2) if fits else None,
                     "判定": pr["verdict"]})
plan = pd.DataFrame(rows)
print("完成对比的任务：%d / %d；判定不做：%s" % (len(EVAL), len(TASKS), summary["skipped"] or "无"))
print(plan.to_string(index=False))
print("新 seed 拟合合计 %.1f 小时（单进程两方模拟的合计计算时间）；实际 / 外推 %.2f ~ %.2f" % (
    plan["新 seed 合计秒数"].sum() / SECONDS_PER_HOUR, plan["实际 / 外推"].min(), plan["实际 / 外推"].max()))

确定性核对（uci_bank · 原版联邦 seed 22 对存档的 seed 11）：逐位相同 False；9 个测试样本打分不同，最大差 0.1463；拟合 151 秒
拟合新 seed [22, 33, 44, 55, 66] 的方法：F_opt_p / F_def


完成对比的任务：9 / 9；判定不做：无
                    任务      方法  前 2 棵逐位一致  单次外推秒数  新 seed 数  新 seed 合计秒数  实际 / 外推 判定
              uci_bank F_opt_p       True    50.4         5        240.4     0.95 ok
              uci_bank   F_def       True   158.2         5        816.1     1.03 ok
        mock_overlap20 F_opt_p       True   220.8         5       1104.2     1.00 ok
        mock_overlap20   F_def       True   298.0         5       1236.9     0.83 ok
                fedads F_opt_p       True   193.8         5        883.3     0.91 ok
                fedads   F_def       True  1731.0         5       8175.3     0.94 ok
        mock_overlap50 F_opt_p       True   547.3         5       2793.9     1.02 ok
        mock_overlap50   F_def       True   380.3         5       1773.2     0.93 ok
          santander_g3 F_opt_p       True   421.3         5       2297.8     1.09 ok
          santander_g3   F_def       True   907.5         5       3972.3     0.88 ok
         orange_upsell F_opt_p       True  1

## 2. 复现

90 个新 seed 合计约 29.5 小时，notebook 不重拟合全部任务。下一格在确定性核对用的任务（UCI）上做三件事：
- 用仓库组件重拟合两个方法的 seed 22，测试打分须与批量运行的存档逐位相同；原版联邦 seed 22 与 seed 11 的差异须与确定性核对相同；
- 从存档打分重算两层 bootstrap 对比（6 seed 与 seed 11 单次）；
- 重算跨任务汇总。

要求与批量运行相同。

In [3]:
REPRO_TOL = 1e-12
NAME = m12["determinism_task"]
SD = SEEDS[0]
res = json.loads((RUN / (NAME + ".json")).read_text(encoding="utf-8"))
sc = SS.MS.Screen(NAME, cfg, PRIV, M12 / "unused")
for m in methods:
    r = SS.fit_seed(P, SS.SC.task_args(P, sc, res, m), SD)
    assert np.array_equal(r["pred"], np.load(SS.seed_path(M12, NAME, m, SD, "npy"))), m
    print("%s · %s seed %d：用仓库组件重拟合（%.0f 秒），测试打分与批量运行的存档逐位相同" % (NAME, m, SD, r["fit_seconds"]))
ref = np.asarray(SS.stored(NAME, SS.DEF, DIRS), np.float32)
new = np.load(SS.seed_path(M12, NAME, SS.DEF, SD, "npy"))
assert int(np.sum(ref != new)) == det["n_rows_differ"]
assert abs(float(np.max(np.abs(ref.astype(float) - new.astype(float)))) - det["max_abs_diff"]) <= REPRO_TOL
print("原版联邦 seed %d 与 seed 11：%d 个测试样本不同，与确定性核对相同" % (SD, det["n_rows_differ"]))
again = SS.evaluate_task(P, sc, res, M12, DIRS, SEEDS, methods)
for k in ("six_seed", "seed11"):
    for f in ("diff", "ci_low", "ci_high", "p", "opt_seed_sd", "def_seed_sd"):
        assert abs(again[k][f] - EVAL[NAME][k][f]) <= REPRO_TOL, (k, f)
again_sum = SS.summarize(EVAL, det, summary["skipped"])
assert [(r["task"], r["holds_six"], r["holds_seed11"]) for r in again_sum["tasks"]] == \
    [(r["task"], r["holds_six"], r["holds_seed11"]) for r in summary["tasks"]]
print("%s：从存档打分重算两层 bootstrap 对比（6 seed 与 seed 11 单次），与批量运行相同（差 ≤ %g）；跨任务汇总重算相同。" % (NAME, REPRO_TOL))

[2026-11-09 08:19:52.356] [info] [bigint_spi.cc:79] The default library used for BigInt operations is openssl


[2026-11-09 08:19:54.234] [info] [thread_pool.cc:30] Create a fixed thread pool with size 1


uci_bank · F_opt_p seed 22：用仓库组件重拟合（47 秒），测试打分与批量运行的存档逐位相同


uci_bank · F_def seed 22：用仓库组件重拟合（151 秒），测试打分与批量运行的存档逐位相同
原版联邦 seed 22 与 seed 11：9 个测试样本不同，与确定性核对相同


uci_bank：从存档打分重算两层 bootstrap 对比（6 seed 与 seed 11 单次），与批量运行相同（差 ≤ 1e-12）；跨任务汇总重算相同。


## 3. 结果

- **I2 + I3 的 6 seed 与 seed 11 单次对比**：配对差、区间、p 与是否成立，两个方法各自的 seed 标准差，以及代理（6 seed）的差与区间。
- **每个（任务，方法）6 个 seed 的 SGB 测试 AUROC**：平均、标准差、最小、最大与极差。
- **17 个任务重算**：列采样的 9 个用本步 6 seed；其余 8 个只读已公开的汇总 CSV（S5.M9 / S5.M2 / S5.M8，seed 11 单次）。
- 汇总 CSV 写入 `results/sgb_seeds_v1/`。

In [4]:
OUT = REPO_ROOT / m12["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
R = m12["round_auc"]
KEY = "%s − %s" % (SS.OPT, SS.DEF)
con = pd.DataFrame([{"dataset": n, "contrast": KEY, "diff_6seed": e["six_seed"]["diff"], "ci_low_6seed": e["six_seed"]["ci_low"],
                     "ci_high_6seed": e["six_seed"]["ci_high"], "p_6seed": e["six_seed"]["p"],
                     "opt_seed_sd": e["six_seed"]["opt_seed_sd"], "def_seed_sd": e["six_seed"]["def_seed_sd"],
                     "diff_seed11": e["seed11"]["diff"], "ci_low_seed11": e["seed11"]["ci_low"],
                     "ci_high_seed11": e["seed11"]["ci_high"], "holds_6seed": e["holds_six"], "holds_seed11": e["holds_seed11"],
                     "same_conclusion": e["holds_six"] == e["holds_seed11"],
                     "proxy_6seed_diff": e["proxy_6seed"]["diff"], "proxy_6seed_ci_low": e["proxy_6seed"]["ci_low"],
                     "proxy_6seed_ci_high": e["proxy_6seed"]["ci_high"]} for n, e in EVAL.items()])
aucs = pd.DataFrame([{"dataset": n, "method": m, "seed": int(sd), "sgb_auroc": a} for n, e in EVAL.items()
                     for m, k in ((SS.OPT, "opt_aucs"), (SS.DEF, "def_aucs")) for sd, a in zip(P.seeds, e["six_seed"][k])])
spread = aucs.groupby(["dataset", "method"])["sgb_auroc"].agg(["mean", "std", "min", "max"]).reset_index()
spread["range"] = spread["max"] - spread["min"]
# 合并已提交的 17 个任务（S5.M9 公开汇总，seed 11 单次）：列采样的 9 个改用本步 6 seed，其余 8 个仍为单次拟合（D-5 / D-8）
prev = pd.read_csv(REPO_ROOT / "modules/m5_modeling/results/sgb_i2i3_v1/i2i3_sgb_all17.csv")
six = con.set_index("dataset")
all17 = pd.DataFrame([{"dataset": r.dataset, "basis": "6 seed（S5.M12）" if r.dataset in six.index else "seed 11 单次（%s）" % r.source,
                       "sgb_diff": six.at[r.dataset, "diff_6seed"] if r.dataset in six.index else r.sgb_diff,
                       "sgb_ci_low": six.at[r.dataset, "ci_low_6seed"] if r.dataset in six.index else r.sgb_ci_low,
                       "sgb_ci_high": six.at[r.dataset, "ci_high_6seed"] if r.dataset in six.index else r.sgb_ci_high,
                       "sgb_holds": bool(six.at[r.dataset, "holds_6seed"]) if r.dataset in six.index else bool(r.sgb_holds),
                       "committed_seed11_diff": r.sgb_diff, "committed_seed11_holds": bool(r.sgb_holds),
                       "proxy_6seed_diff": r.proxy_6seed_diff, "proxy_holds": bool(r.proxy_holds)} for r in prev.itertuples()])
pd.DataFrame([det]).to_csv(OUT / "determinism.csv", index=False, float_format="%.5f")
con.to_csv(OUT / "i2i3_seeds_contrasts.csv", index=False, float_format="%.5f")
aucs.to_csv(OUT / "seed_aurocs.csv", index=False, float_format="%.5f")
spread.to_csv(OUT / "seed_spread.csv", index=False, float_format="%.5f")
all17.to_csv(OUT / "i2i3_sgb_all17_seeds.csv", index=False, float_format="%.5f")
plan.rename(columns={"任务": "dataset", "方法": "method", "前 2 棵逐位一致": "identical_first_two_trees",
                     "单次外推秒数": "projected_fit_seconds", "新 seed 数": "new_seeds", "新 seed 合计秒数": "fit_seconds_new_seeds",
                     "实际 / 外推": "actual_over_projected", "判定": "verdict"}).to_csv(OUT / "probes.csv", index=False, float_format="%.2f")
print("I2 + I3（%s）：两层 bootstrap（测试客户簇 × seed）与 seed 11 单次（同口径）：" % KEY)
print(con.round(R).to_string(index=False))
print("6 seed 成立 %d / %d；seed 11 单次成立 %d / %d；结论改变：%s" % (
    summary["holds_six"], summary["n"], summary["holds_seed11"], summary["n"], summary["changed"] or "无"))
print("每个（任务，方法）6 个 seed 的 SGB 测试 AUROC：")
print(spread.round(R).to_string(index=False))
print("合并后的 17 个任务（列采样的 9 个用 6 seed，其余 8 个为 seed 11 单次）：SGB 上成立 %d / %d（已提交的单次口径 %d / %d）" % (
    all17["sgb_holds"].sum(), len(all17), all17["committed_seed11_holds"].sum(), len(all17)))
print(all17.round(R).to_string(index=False))

I2 + I3（F_opt_p − F_def）：两层 bootstrap（测试客户簇 × seed）与 seed 11 单次（同口径）：
               dataset        contrast  diff_6seed  ci_low_6seed  ci_high_6seed  p_6seed  opt_seed_sd  def_seed_sd  diff_seed11  ci_low_seed11  ci_high_seed11  holds_6seed  holds_seed11  same_conclusion  proxy_6seed_diff  proxy_6seed_ci_low  proxy_6seed_ci_high
              uci_bank F_opt_p − F_def      0.0112        0.0027         0.0196    0.010       0.0009       0.0001       0.0102         0.0009          0.0191         True          True             True            0.0181              0.0101               0.0256
        mock_overlap20 F_opt_p − F_def      0.0062       -0.0056         0.0177    0.290       0.0046       0.0002       0.0137         0.0029          0.0249        False          True            False            0.0081             -0.0050               0.0220
                fedads F_opt_p − F_def      0.0230        0.0112         0.0353    0.000       0.0022       0.0000       0.0203         0.0075  

## 4. 结论

1. **确定性核对未通过**：UCI 原版联邦 seed 22 与 seed 11 在 8,238 个测试样本中 9 个打分不同（最大差 0.146），测试 AUROC 0.7951 → 0.7949。
   列、行采样都为 1 时，seed 仍经被动方分桶洗牌影响增益并列时的切点选择（DR-M5-017 · D-8 的诊断），所以两个方法都用 6 个 seed。
2. **抽查与时长**：
   - 18 个（任务，方法）第一个新 seed 前的前 2 棵，原版与明文打分全部逐位相同，没有任务被跳过；
   - 实际与外推秒数之比 0.83 ~ 1.09；90 个新 seed 分 22 段完成，合计拟合约 29.5 小时；
   - 复现格在 UCI 上重拟合两个方法的 seed 22，与存档逐位相同。
3. **I2 + I3 在真实 SGB 上按 6 seed 计：9 个任务中 5 个成立**（seed 11 单次为 7 个），9 个点估计全部为正：
   - 成立：UCI +0.0112、FedAds +0.0230、Orange 升级 +0.0328、Orange 大版偏好 +0.0220、Orange 大版追加销售 +0.0143；
   - **由成立变为不显著**：mock 重合 20 +0.0062 [−0.0056, +0.0177]（单次 +0.0137）；Santander G3 +0.0053 [−0.0004, +0.0107]（单次 +0.0060）；
   - 两种口径都不显著：mock 重合 50 +0.0051、IEEE 第 1 块 +0.0071。
4. **结论改变的原因不同**：
   - mock 重合 20：seed 11 的优化版联邦（0.9069）是 6 个 seed 中最高的，其余 5 个为 0.8951 ~ 0.9020，单次结论高估了幅度；
   - Santander G3：幅度基本不变，单次区间下限原本只有 +0.0002，计入训练随机性后含 0。
5. **训练随机性的量级**（单个方法 SGB 测试 AUROC 的 6 seed 标准差）：
   - 优化版联邦（列采样 < 1）0.0009 ~ 0.0046，mock 重合 20 最大（极差 0.012）；
   - 原版联邦（列、行采样都为 1）0.0000 ~ 0.0003，FedAds、mock 重合 50、Santander G3、IEEE 第 1 块的标准差保留 4 位小数为 0。
6. **17 个任务重算：13 / 17 成立**（已提交的单次口径 15 / 17）。
   - 其余 8 个任务的优化版联邦列、行采样都为 1，仍为 seed 11 单次，8 个都成立；它们同样含分桶洗牌这类随机性。
   - 以原版联邦 6 seed 标准差 ≤ 0.0003 为量级参考，这 8 个任务的区间下限最小为 +0.0025（mock 重合 80）；这只是参考，没有对这 8 个任务做多 seed。
   - 与代理 6 seed 的结论一致数：9 个任务中两种口径都是 7 / 9，17 个任务合计都是 14 / 17。

**对报告的意义**：I2 + I3 的 SGB 证据改为 6 seed 口径后，17 个任务中 13 个成立；列采样配置的 SGB 单次结论要按训练随机性折扣，小幅差（约 +0.005）的任务不再算成立。

**边界**：
- 6 个 seed 是训练随机性的有限抽样；p 未做多重比较校正（本阶段未预先写定 Holm）；
- 其余 8 个任务未做多 seed，扩到这 8 个任务需所有者另行授权；
- 完整拟合与原版的逐位一致只在 S5.M7 的 6 个任务上证实，本步新 seed 是前 2 棵抽查；
- 明文直方图只在实验代码里替换一个方法，用于正式联调需经安全评审；
- 公开数据上的开发证据，不换算成汇丰的命中率或收益；技术侧自评，未经法务复核（W-001）。